In [ ]:
##creating a model to suggest a vaccine given symptoms and age

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score


In [ ]:
df = pd.read_csv('/Users/rishabhsharma/Documents/GitHub/Covid-19-application/data/2021VAERSDATA.csv',encoding='latin-1')
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.drop(['RECVDATE', 'CAGE_YR', 'CAGE_MO', 'RPT_DATE', 'DIED', 'DATEDIED', 'ER_VISIT', 'HOSPITAL', 'HOSPDAYS', 'X_STAY', 'DISABLE'
            ,'RECOVD', 'VAX_DATE', 'ONSET_DATE', 'NUMDAYS', 'LAB_DATA', 'V_ADMINBY', 'V_FUNDBY','SPLTTYPE','FORM_VERS', 'TODAYS_DATE', 'OFC_VISIT', 'ER_ED_VISIT'
            ,'L_THREAT', 'PRIOR_VAX', 'BIRTH_DEFECT'], axis=1, inplace=True)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df['STATE'].fillna(df['STATE'].mode()[0], inplace=True)
df['AGE_YRS'].fillna(df['AGE_YRS'].mode()[0], inplace=True)

In [ ]:
df['SYMPTOM_TEXT'].fillna(df['SYMPTOM_TEXT'].mode()[0], inplace=True)

In [ ]:
df['CUR_ILL'].value_counts() ##illness at the time of vaccination

## filling the missing values with "no illness at the time of vaccination"
df['CUR_ILL'].fillna("No illness at the time of vaccination", inplace=True)

In [ ]:
df['OTHER_MEDS'].fillna("No medication", inplace=True)  ## filling the missing values with "no medication"

In [ ]:
df['HISTORY'] = df['HISTORY'].astype(str) + df['ALLERGIES'].astype(str)

In [ ]:
df.drop('ALLERGIES', axis=1, inplace=True)

In [ ]:
##changing Na, anNa, none to None in all the columns

df.replace('Na', 'None', inplace=True)
df.replace('anNa', 'None', inplace=True)
df.replace('none', 'None', inplace=True)
df.replace('NONE', 'None', inplace=True)

In [ ]:
## replace nannan, NoneNone, nonenone to No history and allergies in history column

df['HISTORY'].replace(['nannan','NoneNone', 'nonenone', 'nanNa', 'NanNa'], 'No history and allergies', inplace=True)

In [ ]:
df['CUR_ILL'] = df['CUR_ILL'].apply(lambda x: 'None' if x in df['CUR_ILL'].value_counts()[0:16].index else x)

In [ ]:
df.drop([], axis=1, inplace=True)

In [ ]:
## counting NAN values in each column

df.isnull().sum()

In [ ]:
df.head()

In [ ]:
df1 = pd.read_csv('/Users/rishabhsharma/Documents/GitHub/Covid-19-application/data/2021VAERSVAX.csv')

In [ ]:
df1.head()

In [ ]:
df1.isnull().sum()

In [ ]:
df1.columns

In [ ]:
df1.drop(['VAX_TYPE', 'VAX_MANU', 'VAX_LOT', 'VAX_DOSE_SERIES',
       'VAX_ROUTE', 'VAX_SITE'], axis = 1, inplace = True)

In [ ]:
df.head()

In [ ]:
df1.head()

In [ ]:
##merge the two dataframes on VAERS_ID

df = pd.merge(df, df1, on = 'VAERS_ID', how = 'inner')

In [ ]:
df.shape

In [ ]:
df.isnull().any()

In [ ]:
df.drop(['VAERS_ID'], axis = 1, inplace = True)

In [ ]:
df.head()

In [ ]:
df['HISTORY'] = df['HISTORY'].apply(lambda x: x[4:] if x.startswith('None') else x)

In [ ]:
df.head()

In [ ]:
df['VAX_NAME'].value_counts()[0:10]

In [ ]:
##plot the top 10 vaccines using barplot

plt.figure(figsize=(10,5))
sns.barplot(x = df['VAX_NAME'].value_counts()[0:10].index, y = df['VAX_NAME'].value_counts()[0:10].values)

In [ ]:
## if VAX_NAME is not in first 2 vaccines then delete the row

df = df[df['VAX_NAME'].isin(df['VAX_NAME'].value_counts()[0:2].index)]

In [ ]:
sns.barplot(x = df['VAX_NAME'].value_counts()[0:10].index, y = df['VAX_NAME'].value_counts()[0:10].values)

In [ ]:
df.head()

In [ ]:
## convert OTHER_MEDS to space if it is none, CUR_ILL to space if it is none, and HISTORY to space if it is no history and allergies

df['OTHER_MEDS'] = df['OTHER_MEDS'].apply(lambda x: ' ' if x == 'None' else x)
df['CUR_ILL'] = df['CUR_ILL'].apply(lambda x: ' ' if x == 'None' else x)
df['HISTORY'] = df['HISTORY'].apply(lambda x: ' ' if x == 'history allergies' else x)

In [ ]:
df['HISTORY'].value_counts()[:14]